# data-cleaning_012

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (49).ipynb`

| Field | Value |
|---|---|
| Section | `data_cleaning` |
| Market | `options` |
| Trading style | `unknown` |
| Timeframe | `3m` |
| Code cells | 20 |
| Detected functions | plot_trade_summary, check_margin_call, monte_carlo_simulation |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/tradebook_all_tickers.csv')
df

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Set Seaborn style for consistent Matplotlib aesthetics
sns.set(style='whitegrid')

# Prepare the DataFrame: Convert datetime columns and sort by 'Entry Time'
df = df.copy()  # Avoid modifying the original DataFrame
df['Entry Time'] = pd.to_datetime(df['Entry Time'], utc=True)  # Auto-detect format and timezone
df['Exit Time'] = pd.to_datetime(df['Exit Time'], utc=True)
df = df.sort_values('Entry Time')

# Histogram of PnL distribution with KDE-like appearance
fig = px.histogram(df, x='PnL', nbins=300, marginal='rug',
                   title='Distribution of PnL',
                   opacity=0.75,
                   color_discrete_sequence=['forestgreen'])

fig.update_layout(
    xaxis_title='PnL',
    yaxis_title='Frequency',
    bargap=0.05,
    template='plotly_white'
)

fig.show()

# Ensure 'Action' is string/categorical
df['Action'] = df['Action'].astype(str)

# Create faceted histogram
fig = px.histogram(
    df,
    x='PnL',
    facet_col='Action',
    color='Action',
    nbins=300,
    opacity=0.75,
    title='PnL Distribution by Action',
    color_discrete_sequence=px.colors.qualitative.Set2
)

fig.update_layout(
    template='plotly_white',
    showlegend=False,
    height=400  # adjust height as needed
)

# Optional: improve spacing between subplots
fig.update_traces(marker_line_width=0.5, marker_line_color='black')
fig.update_xaxes(matches=None, showticklabels=True)
fig.update_yaxes(showgrid=True)

fig.show()

# --- Visualization 5: Interactive Trade Summary by Ticker ---
def plot_trade_summary(df_df):
    df_df = df_df.copy()
    df_df['Win'] = df_df['PnL'] > 0
    df_df['Loss'] = df_df['PnL'] < 0

    summary = df_df.groupby('Ticker').agg(
        Total_Trades=('Ticker', 'count'),
        Winning_Trades=('Win', 'sum'),
        Losing_Trades=('Loss', 'sum')
    ).reset_index()

    summary_melted = summary.melt(
        id_vars='Ticker',
        value_vars=['Winning_Trades', 'Losing_Trades'],
        var_name='Trade Type',
        value_name='Count'
    )

    fig = px.bar(
        summary_melted,
        x='Ticker',
        y='Count',
        color='Trade Type',
        barmode='group',
        title='Trade Summary by Ticker',
        labels={'Count': 'Number of Trades', 'Ticker': 'Stock Ticker'},
        text_auto=True,
        color_discrete_map={'Winning_Trades': 'green', 'Losing_Trades': 'red'}
    )
    fig.update_layout(
        xaxis_tickangle=-45,
        title_font_size=16,
        xaxis_title_font_size=14,
        yaxis_title_font_size=14,
        legend_title_text='Trade Outcome'
    )
    fig.show()

plot_trade_summary(df)

# --- Visualization 6: Bar Plot for Sum of PnL by Ticker ---
pnl_by_ticker = df.groupby('Ticker')['PnL'].sum().sort_values(ascending=False)
plt.figure(figsize=(15, 6))
pnl_by_ticker.plot(kind='bar', color='mediumpurple', edgecolor='black')
plt.title('Total PnL by Ticker', fontsize=14, pad=10)
plt.xlabel('Ticker', fontsize=12)
plt.ylabel('Total PnL', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# --- Visualization 7: Interactive PnL by Weekday ---
df['Weekday'] = df['Exit Time'].dt.day_name()
weekday_pnl = df.groupby('Weekday')['PnL'].sum().reset_index()
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekday_pnl['Weekday'] = pd.Categorical(weekday_pnl['Weekday'], categories=weekday_order, ordered=True)
weekday_pnl = weekday_pnl.sort_values('Weekday')
weekday_pnl['PnL_Type'] = weekday_pnl['PnL'].apply(lambda x: 'Positive' if x > 0 else 'Negative')

fig = px.bar(
    weekday_pnl,
    x='Weekday',
    y='PnL',
    title='Total PnL by Weekday',
    labels={'PnL': 'Total PnL', 'Weekday': 'Day of Week'},
    color='PnL_Type',
    color_discrete_map={'Positive': 'green', 'Negative': 'red'},
    text_auto=True
)
fig.update_layout(
    title_font_size=16,
    xaxis_title_font_size=14,
    yaxis_title_font_size=14,
    legend_title_text='PnL Type'
)
fig.show()

In [ ]:
summary = df['Action'].value_counts().reset_index()
summary.columns = ['Action', 'Count']
print("Number of unique Actions:", len(summary))
print(summary)

In [ ]:
import plotly.graph_objects as go
import pandas as pd

# Assuming 'df' has already been sorted and includes 'PnL'
df = df.sort_values('Entry Time')
df['Cumulative PnL'] = df['PnL'].cumsum()
df['Peak'] = df['Cumulative PnL'].cummax()
df['Drawdown'] = ((df['Cumulative PnL'] - df['Peak']) / 1000000) * 100# As percent

# Create figure with subplots
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.1,
    subplot_titles=("Portfolio Balance Over Time", "Drawdown (%)")
)

# 1. Balance Over Time
fig.add_trace(
    go.Scatter(x=df['Entry Time'], y=df['Cumulative PnL'], mode='lines', name='Balance', line=dict(color='purple')),
    row=1, col=1
)

# 2. Drawdown Percentage
fig.add_trace(
    go.Scatter(x=df['Entry Time'], y=df['Drawdown'], mode='lines', name='Drawdown %', line=dict(color='red')),
    row=2, col=1
)

# Update layout
fig.update_layout(
    height=700,
    width=1000,
    title_text="Portfolio Performance Overview (Interactive)",
    showlegend=True,
    template='plotly_white'
)

# Axis labels
fig.update_yaxes(title_text="Balance", row=1, col=1)
fig.update_yaxes(title_text="Drawdown (%)", row=2, col=1)
fig.update_xaxes(title_text="Entry Time", row=2, col=1)

fig.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors as mcolors

# Convert 'Entry Time' to datetime
df['Entry Time'] = pd.to_datetime(df['Entry Time'])

# Extract year-month
df['YearMonth'] = df['Entry Time'].dt.to_period('M')

# Convert PnL to percentage based on ₹1,00,000 capital
df['PnL %'] = (df['PnL'] / 100000) * 100

# Group by Ticker and Month, then pivot
pnl_pct_matrix = df.groupby(['Ticker', 'YearMonth'])['PnL %'].sum().unstack().fillna(0)
pnl_pct_matrix.columns = pnl_pct_matrix.columns.astype(str)

pnl_pct_matrix.to_csv("monthly_pnl_percentage_matrix.csv")

# Create red-white-green colormap centered at 0
cmap = mcolors.LinearSegmentedColormap.from_list("red_white_green", ["red", "white", "green"])

# Find the max absolute value for symmetrical color scaling
vmax = abs(pnl_pct_matrix.values).max()

# Plot heatmap
plt.figure(figsize=(20, 5))
sns.heatmap(pnl_pct_matrix, annot=True, fmt=".2f", cmap=cmap, center=0,
            vmin=-vmax, vmax=vmax, linewidths=0.1, linecolor='gray', annot_kws={"size": 10})

plt.title('Monthly PnL % Per Ticker ($100K capital)', fontsize=20, pad=5)
plt.xlabel('Month')
plt.ylabel('Ticker')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure datetime format
df['Entry Time'] = pd.to_datetime(df['Entry Time'])

# Monthly return % based on ₹1L capital
df['YearMonth'] = df['Entry Time'].dt.to_period('M')
df['PnL %'] = (df['PnL'] / 300000) * 100

# Group by Ticker and Month, sum PnL % for monthly returns
monthly_returns = df.groupby(['Ticker', 'YearMonth'])['PnL %'].sum().reset_index()

# Pivot: Ticker x Month
returns_matrix = monthly_returns.pivot(index='Ticker', columns='YearMonth', values='PnL %').sort_index(axis=1)

# Compute rolling Sharpe (mean / std over 3 months)
rolling_mean = returns_matrix.rolling(window=3, axis=1).mean()
rolling_std = returns_matrix.rolling(window=3, axis=1).std()
rolling_sharpe = rolling_mean / rolling_std

rolling_sharpe.to_csv("rolling_3M_sharpe_ratio.csv")

# Plot heatmap
plt.figure(figsize=(50, 5))
sns.heatmap(rolling_sharpe, annot=True, fmt=".2f", cmap='RdYlGn', center=0,
            linewidths=0.5, linecolor='gray', cbar_kws={'label': 'Rolling 3M Sharpe'}, annot_kws={"size": 10})
plt.title('Rolling 3-Month Sharpe Ratio per Ticker', fontsize=16, pad=12)
plt.xlabel('Month')
plt.ylabel('Ticker')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Convert 'Entry Time' to datetime and extract the hour
df['Hour'] = pd.to_datetime(df['Entry Time']).dt.hour

# Categorize trades as Win or Loss
df['Trade Type'] = df['PnL'].apply(lambda x: 'Win' if x > 0 else 'Loss')

# Group by hour to get total PnL per hour
pnl_per_hour = df.groupby('Hour')['PnL'].sum().reset_index()

# Create subplot figure
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1,
                    subplot_titles=("Number of Win & Loss Trades Per Hour", "Total PnL Per Hour"))

# Histogram for trade count
histogram = px.histogram(df, x='Hour', color='Trade Type', barmode='group')

# Bar chart for total PnL per hour
pnl_bar = go.Bar(x=pnl_per_hour['Hour'], y=pnl_per_hour['PnL'], name='Total PnL', marker_color='blue')

# Add traces to the subplot
for trace in histogram.data:
    fig.add_trace(trace, row=1, col=1)

fig.add_trace(pnl_bar, row=2, col=1)

# Update layout
fig.update_layout(title_text="Trade Analysis Per Hour", xaxis_title="Hour of Day", height=700, width= 1250)

# Show figure
fig.show()


In [ ]:
import pandas as pd

# Prepare data
df['Entry Time'] = pd.to_datetime(df['Entry Time'])
pnl_matrix = df.pivot_table(index='Entry Time', columns='Ticker', values='PnL')
traded = pnl_matrix.notnull() & (pnl_matrix != 0)

num_traded_tickers = traded.sum(axis=1)
total_times = len(num_traded_tickers)

percentages = {}
for i in range(traded.shape[1], 0, -1):  # from max tickers down to 1
    count_i = (num_traded_tickers == i).sum()
    percentages[i] = (count_i / total_times) * 100

# Print output in requested format
for n in range(traded.shape[1], 0, -1):
    pct = percentages[n]
    print(f"{pct:.2f}% of time all {n} tickers are traded at the same time")


In [ ]:
capital = 100_000  # Your starting capital
leverage = 10      # Max leverage used
max_dd = 0.12      # Historical max leveraged drawdown (4%)
margin_call_threshold = 0.25  # Broker margin call at 25% margin loss

# Function to check margin call risk at given drawdown
def check_margin_call(drawdown, capital, leverage, margin_call_threshold):
    # Effective loss on capital due to leverage
    leveraged_loss = capital * drawdown

    # Equity remaining after loss
    equity = capital - leveraged_loss

    # Margin used (assuming full leverage)
    margin_used = capital / leverage

    # Margin call if equity < margin_call_threshold * margin_used
    margin_call = equity < margin_call_threshold * margin_used

    return margin_call, equity

# Test for different extreme drawdowns (5%, 10%, 20%, 30%)
extreme_drawdowns = [0.05, 0.10, 0.20, 0.30]

for dd in extreme_drawdowns:
    mc, eq = check_margin_call(dd, capital, leverage, margin_call_threshold)
    status = "MARGIN CALL" if mc else "No margin call"
    print(f"Drawdown: {dd*100:.1f}% -> Equity: ${eq:,.2f} -> {status}")


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import calendar

# Ensure 'Entry Time' is in datetime format
df['Entry Time'] = pd.to_datetime(df['Entry Time'])

# Extract month and year
df['Year'] = df['Entry Time'].dt.year
df['Month'] = df['Entry Time'].dt.month

# Monthly total profit percent
monthly_profit = df.groupby(['Year', 'Month'])['PnL'].sum().reset_index()
monthly_profit['Profit Percent'] = (monthly_profit['PnL'] / 1000000) * 100

# Yearly return
yearly_return = df.groupby('Year')['PnL'].sum().reset_index()
yearly_return['Profit Percent'] = (yearly_return['PnL'] / 1000000) * 100

# Pivot table for heatmap
pivot_table = monthly_profit.pivot(index='Year', columns='Month', values='Profit Percent')
pivot_table.columns = [calendar.month_abbr[i] for i in pivot_table.columns]

# 📊 Heatmap
heatmap = px.imshow(
    pivot_table,
    text_auto=".2f",
    color_continuous_scale="YlGnBu",
    labels={"color": "Profit %"},
    title="Monthly Total Profit Percentage",
)

heatmap.update_layout(
    width=1200,
    height=500,
    xaxis_title="Month",
    yaxis_title="Year",
    xaxis=dict(tickmode="array", tickvals=list(range(len(pivot_table.columns))), ticktext=list(pivot_table.columns)),
    yaxis=dict(tickmode="array", tickvals=pivot_table.index),
    coloraxis_colorbar=dict(title="Profit %"),
)

heatmap.show()

# 📊 Bar Chart for Yearly Return
bar_chart = go.Figure()

bar_chart.add_trace(go.Bar(
    x=yearly_return['Year'],
    y=yearly_return['Profit Percent'],
    marker=dict(color="skyblue"),
    text=yearly_return['Profit Percent'].round(2),
    textposition='outside',
))

bar_chart.update_layout(
    width=1200,
    height=500,
    title="Yearly Total Profit Percentage",
    xaxis_title="Year",
    yaxis_title="Yearly Return (%)",
    bargap=0.2,
    template="plotly_white"
)

bar_chart.show()


In [ ]:
import pandas as pd
import numpy as np

# === Assuming df is already loaded ===
# Convert Entry/Exit Time to datetime if not already
df['Entry Time'] = pd.to_datetime(df['Entry Time'])
df['Exit Time'] = pd.to_datetime(df['Exit Time'])

# === DAILY SHARPE CALCULATION ===
df['Trade Date'] = df['Entry Time'].dt.date
daily_pnl = df.groupby('Trade Date')['PnL'].sum()

mean_daily_return = daily_pnl.mean()
std_daily_return = daily_pnl.std()
sharpe_daily = (mean_daily_return / std_daily_return) * np.sqrt(365) if std_daily_return != 0 else np.nan

# === BASIC METRICS ===
total_trades = len(df)
winning_trades = df[df['PnL'] > 0]
losing_trades = df[df['PnL'] < 0]
win_rate = len(winning_trades) / total_trades
loss_rate = len(losing_trades) / total_trades
avg_win = winning_trades['PnL'].mean()
avg_loss = losing_trades['PnL'].mean()
avg_pnl = df['PnL'].mean()
total_pnl = df['PnL'].sum()
profit_factor = winning_trades['PnL'].sum() / abs(losing_trades['PnL'].sum()) if len(losing_trades) > 0 else np.inf
expectancy = (win_rate * avg_win) + (loss_rate * avg_loss)

# === CUMULATIVE PNL & DRAWDOWN ===
df['Cumulative PnL'] = df['PnL'].cumsum()
cumulative_pnl = df['Cumulative PnL']
running_max = cumulative_pnl.cummax()
drawdown = running_max - cumulative_pnl
max_drawdown = drawdown.max()

# === Drawdown Duration ===
drawdown_periods = []
start = None

for i in range(len(cumulative_pnl)):
    if cumulative_pnl[i] < running_max[i]:
        if start is None:
            start = i
    else:
        if start is not None:
            duration = (df.iloc[i]['Entry Time'] - df.iloc[start]['Entry Time']).total_seconds() / 60
            drawdown_periods.append(duration)
            start = None
if start is not None:
    duration = (df.iloc[-1]['Exit Time'] - df.iloc[start]['Entry Time']).total_seconds() / 60
    drawdown_periods.append(duration)

max_drawdown_duration = max(drawdown_periods) if drawdown_periods else 0
avg_drawdown_duration = np.mean(drawdown_periods) if drawdown_periods else 0

# === Sortino Ratio ===
downside_returns = daily_pnl[daily_pnl < 0]
downside_std = downside_returns.std() if len(downside_returns) > 0 else np.nan
sortino_ratio = (mean_daily_return / downside_std) * np.sqrt(365) if downside_std != 0 else np.nan

# === Calmar Ratio ===
annualized_return = mean_daily_return * 365
calmar_ratio = annualized_return / max_drawdown if max_drawdown != 0 else np.nan

# === Holding Time ===
df['Holding Time'] = (df['Exit Time'] - df['Entry Time']).dt.total_seconds() / 60
avg_holding_time = df['Holding Time'].mean()

# === Total Time Span ===
start_time = df['Entry Time'].min()
end_time = df['Exit Time'].max()
total_time_span = (end_time - start_time).total_seconds() / 60  # in minutes

# === TRUE TIME INVESTED WITHOUT OVERLAP ===
intervals = list(zip(df['Entry Time'], df['Exit Time']))
merged_intervals = []

for start, end in sorted(intervals):
    if not merged_intervals or start > merged_intervals[-1][1]:
        merged_intervals.append([start, end])
    else:
        merged_intervals[-1][1] = max(merged_intervals[-1][1], end)

true_time_invested = sum([(end - start).total_seconds() / 60 for start, end in merged_intervals])  # in minutes
true_investment_ratio = true_time_invested / total_time_span if total_time_span != 0 else np.nan

# === Average Number of Trades Per Day ===
unique_days = df['Trade Date'].nunique()
avg_trades_per_day = total_trades / unique_days if unique_days != 0 else np.nan

# === OUTPUT METRICS ===
metrics = {
    "Total Trades": total_trades,
    "Win Rate": win_rate,
    "Loss Rate": loss_rate,
    "Average PnL": avg_pnl,
    "Total PnL": total_pnl,
    "Profit Factor": profit_factor,
    "Expectancy": expectancy,
    "Max Drawdown": max_drawdown,
    "Sharpe Ratio (Daily)": sharpe_daily,
    "Sortino Ratio": sortino_ratio,
    "Calmar Ratio": calmar_ratio,
    "Average Holding Time (mins)": avg_holding_time,
    "True Time Invested (mins)": true_time_invested,
    "Total Time Span (mins)": total_time_span,
    "True Investment Ratio": true_investment_ratio,
    "Max Drawdown Duration (mins)": max_drawdown_duration,
    "Avg Drawdown Duration (mins)": avg_drawdown_duration,
    "Average Trades Per Day": avg_trades_per_day
}

# === Print the Metrics ===
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"{k}: {v:.2f}")
    else:
        print(f"{k}: {v}")


In [ ]:
# 📦 Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 📂 Load your df
df['Entry Time'] = pd.to_datetime(df['Entry Time'])
df['Exit Time'] = pd.to_datetime(df['Exit Time'])

# 🛠 Prepare data: Create Exit Date
df['Exit Date'] = df['Exit Time'].dt.date

# 🛠 Daily Aggregation
daily = df.groupby('Exit Date').agg({
    'PnL': 'sum',
    'Entry Price': 'mean',
    'Exit Price': ['max', 'min']
})
daily.columns = ['PnL', 'Entry Price', 'Max Exit Price', 'Min Exit Price']

# 📈 Calculate Daily Range % (simple volatility proxy)
daily['Daily Range %'] = (daily['Max Exit Price'] - daily['Min Exit Price']) / daily['Entry Price'] * 100

# 🔥 Define Volatility Clusters
vol_threshold = daily['Daily Range %'].median()
daily['Volatility Regime'] = np.where(daily['Daily Range %'] > vol_threshold, 'High Volatility', 'Low Volatility')

# 🔗 Merge Volatility Regime back to df
df = df.merge(daily[['Volatility Regime']], left_on=df['Exit Time'].dt.date, right_index=True)

# 📊 Analyze Performance by Volatility
volatility_performance = df.groupby('Volatility Regime').agg({
    'PnL': ['mean', 'sum', 'count'],
    'Direction': 'count'
})
volatility_performance.columns = ['Avg PnL', 'Total PnL', 'Number of Trades', 'Number of Trades Direction']

# 🎯 Win Rate calculation
win_rate = df.assign(Win = df['PnL'] > 0).groupby('Volatility Regime')['Win'].mean() * 100

# 🎨 Plotting - Vertically stacked, no grid lines
fig, axes = plt.subplots(2, 1, figsize=(5, 7))  # 2 rows, 1 column

# 📈 Top: Average PnL per Trade
volatility_performance['Avg PnL'].plot(kind='bar', ax=axes[0], title='Average PnL per Trade by Volatility Regime', color=['steelblue', 'salmon'])
axes[0].set_ylabel('Average PnL (₹)')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', labelrotation=0)
axes[0].grid(False)  # 🔴 Remove grid

# 📈 Bottom: Win Rate in each Volatility Regime
win_rate.plot(kind='bar', ax=axes[1], title='Win Rate by Volatility Regime', color=['seagreen', 'tomato'])
axes[1].set_ylabel('Win Rate (%)')
axes[1].set_xlabel('Volatility Regime')
axes[1].tick_params(axis='x', labelrotation=0)
axes[1].grid(False)  # 🔴 Remove grid

plt.tight_layout()
plt.show()

# 🖨 Print the table
print(volatility_performance)
print("\nWin Rates (%):")
print(win_rate)


In [ ]:
import pandas as pd
import plotly.express as px

# Ensure datetime format
df['Entry Time'] = pd.to_datetime(df['Entry Time'])

# Pivot: rows = Entry Time, columns = Ticker, values = PnL
pnl_matrix = df.pivot_table(index='Entry Time', columns='Ticker', values='PnL')

# Drop columns with very few non-null values to clean matrix
pnl_matrix = pnl_matrix.dropna(thresh=10, axis=1)

# Fill missing values (optional but useful)
pnl_matrix = pnl_matrix.fillna(0)

# Compute correlation matrix
pnl_corr = pnl_matrix.corr()

# Plot correlation heatmap using Plotly
fig = px.imshow(
    pnl_corr,
    title='PnL Correlation Between Tickers (Aligned by Entry Time)',
    color_continuous_scale='RdBu',
    zmin=-1, zmax=1
)
fig.update_layout(width=900, height=900)
fig.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load your tradebook data
# Example: df = pd.read_csv("tradebook.csv")

# Step 1: Ensure 'Exit Time' is datetime and extract date
df["Exit Time"] = pd.to_datetime(df["Exit Time"])
df["Date"] = df["Exit Time"].dt.date

# Step 2: Aggregate daily PnL and normalize by fixed ₹10 lakh capital
daily_pnl = df.groupby("Date")["PnL"].sum()
daily_returns = daily_pnl / 1_000_000  # Normalized return per ₹10 lakh

# Step 3: Monte Carlo Simulation Function
def monte_carlo_simulation(pnl_series, num_simulations=1_00_000, num_days=None):
    if num_days is None:
        num_days = len(pnl_series)

    # Sample daily returns with replacement
    simulations = np.random.choice(pnl_series, size=(num_simulations, num_days), replace=True)
    cumulative_returns = np.cumsum(simulations, axis=1)
    return cumulative_returns

# Step 4: Run the simulation
simulations = monte_carlo_simulation(daily_returns, num_simulations=1_00_000)

# Step 5: Plot first 500 simulation paths
plt.figure(figsize=(15, 6))
for sim in simulations[:500]:  # Corrected to 500
    plt.plot(sim, color="gray", alpha=0.1)
plt.title("Monte Carlo Simulations of Strategy PnL (500 paths)")
plt.xlabel("Days")
plt.ylabel("Cumulative Return (Normalized to ₹10L)")
plt.grid(True)
plt.tight_layout()
plt.show()

# Step 6: Print Summary Statistics
final_returns = simulations[:, -1]
print("Mean Final Return:", round(np.mean(final_returns), 2))
print("5th Percentile Return (Worst Case):", round(np.percentile(final_returns, 5), 2))
print("95th Percentile Return (Best Case):", round(np.percentile(final_returns, 95), 2))
print("Minimum Return:", round(np.min(final_returns), 2))
print("Maximum Return:", round(np.max(final_returns), 2))


In [ ]:
plt.hist(final_returns, bins=100, color="skyblue", edgecolor="black")
plt.title("Distribution of Final Returns")
plt.xlabel("Final Return (Normalized)")
plt.ylabel("Frequency")
plt.grid(True)
plt.show()


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import calendar

# Example weights (adjust according to your strategy)
ticker_weights = {
    'BTCUSDT': 0.12,
    'ETHUSDT': 0.13,
    'SOLUSDT': 0.21,
    'XRPUSDT': 0.14,
    'BNBUSDT': 0.10,
    'DOGEUSDT': 0.16,
    'ADAUSDT': 0.14,
}

# Ensure 'Entry Time' is datetime
df['Entry Time'] = pd.to_datetime(df['Entry Time'])

# Assign weights
df['Weight'] = df['Ticker'].map(ticker_weights)

# Apply weights to PnL
df['Weighted_PnL'] = df['PnL'] * df['Weight']

# Extract month and year
df['Year'] = df['Entry Time'].dt.year
df['Month'] = df['Entry Time'].dt.month

# Monthly total profit percent (Weighted)
monthly_profit = df.groupby(['Year', 'Month'])['Weighted_PnL'].sum().reset_index()
monthly_profit['Profit Percent'] = (monthly_profit['Weighted_PnL'] / 500000) * 100

# Yearly return (Weighted)
yearly_return = df.groupby('Year')['Weighted_PnL'].sum().reset_index()
yearly_return['Profit Percent'] = (yearly_return['Weighted_PnL'] / 500000) * 100

# Pivot for heatmap
pivot_table = monthly_profit.pivot(index='Year', columns='Month', values='Profit Percent')
pivot_table.columns = [calendar.month_abbr[i] for i in pivot_table.columns]

# 📊 Heatmap
heatmap = px.imshow(
    pivot_table,
    text_auto=".2f",
    color_continuous_scale="YlGnBu",
    labels={"color": "Profit %"},
    title="Monthly Total Profit Percentage (Weighted by Ticker)",
)

heatmap.update_layout(
    width=1200,
    height=500,
    xaxis_title="Month",
    yaxis_title="Year",
    xaxis=dict(tickmode="array", tickvals=list(range(len(pivot_table.columns))), ticktext=list(pivot_table.columns)),
    yaxis=dict(tickmode="array", tickvals=pivot_table.index),
    coloraxis_colorbar=dict(title="Profit %"),
)

heatmap.show()

# 📊 Bar Chart for Yearly Return
bar_chart = go.Figure()

bar_chart.add_trace(go.Bar(
    x=yearly_return['Year'],
    y=yearly_return['Profit Percent'],
    marker=dict(color="skyblue"),
    text=yearly_return['Profit Percent'].round(2),
    textposition='outside',
))

bar_chart.update_layout(
    width=1200,
    height=500,
    title="Yearly Total Profit Percentage (Weighted)",
    xaxis_title="Year",
    yaxis_title="Yearly Return (%)",
    bargap=0.2,
    template="plotly_white"
)

bar_chart.show()


In [ ]:
import pandas as pd
import plotly.express as px

# === 1. Define capital ===
tickers_traded = df['Ticker'].nunique()  # or set manually = 5
capital_per_ticker = 100000
total_daily_capital = tickers_traded * capital_per_ticker

# === 2. Set index and calculate daily strategy returns ===
df = df.set_index('Exit Time')
daily_strategy_pnl = df['PnL'].resample('D').sum()
daily_strategy = (daily_strategy_pnl / total_daily_capital) * 100  # as % return
daily_strategy.name = 'Strategy_Returns'

# === 3. Extract BTCUSDT daily returns ===
btc_df = df[df['Ticker'] == 'BTCUSDT'][['Exit Price']]
btc_df = btc_df[~btc_df.index.duplicated(keep='last')]

btc_daily_close = btc_df['Exit Price'].resample('D').last()
btc_daily_returns = btc_daily_close.pct_change() * 100
btc_daily_returns.name = 'BTCUSDT_Returns'

# === 4. Merge ===
daily_df = pd.concat([daily_strategy, btc_daily_returns], axis=1).dropna()

# === 5. Plotly Scatterplot ===
fig = px.scatter(
    daily_df,
    x='BTCUSDT_Returns',
    y='Strategy_Returns',
    hover_data={'BTCUSDT_Returns': ':.2f', 'Strategy_Returns': ':.2f'},
    labels={
        "BTCUSDT_Returns": "BTCUSDT Daily Returns (%)",
        "Strategy_Returns": "Strategy Daily Returns (%)"
    },
    title="Strategy Daily % Returns vs BTCUSDT Daily % Returns",
    opacity=0.7,
    template='plotly_white'
)

fig.add_shape(type='line', x0=0, x1=0, y0=daily_df['Strategy_Returns'].min(), y1=daily_df['Strategy_Returns'].max(),
              line=dict(color='gray', dash='dash'))
fig.add_shape(type='line', y0=0, y1=0, x0=daily_df['BTCUSDT_Returns'].min(), x1=daily_df['BTCUSDT_Returns'].max(),
              line=dict(color='gray', dash='dash'))

fig.update_layout(showlegend=False)
fig.show()

# === 6. Correlation ===
print("Correlation:", daily_df['BTCUSDT_Returns'].corr(daily_df['Strategy_Returns']))


In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import calendar

# Remove trades at exactly 09:15
df = df[df['Entry Time'].dt.time != pd.to_datetime("09:15").time()]

# Sort and calculate metrics
df = df.sort_values('Entry Time')
df['Cumulative PnL'] = df['PnL'].cumsum()
df['Peak'] = df['Cumulative PnL'].cummax()
df['Drawdown'] = ((df['Cumulative PnL'] - df['Peak']) / 1000000) * 100  # % drawdown

# ------------------------------
# 1. Portfolio Balance & Drawdown
# ------------------------------
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.1,
    subplot_titles=("Portfolio Balance Over Time", "Drawdown (%)")
)

# Portfolio balance line
fig.add_trace(
    go.Scatter(
        x=df['Entry Time'], y=df['Cumulative PnL'],
        mode='lines+markers',
        name='Balance',
        line=dict(color='purple'),
        text=df['Cumulative PnL'].round(2).astype(str),
        hovertemplate='Date: %{x}<br>Balance: ₹%{text}'
    ),
    row=1, col=1
)

# Drawdown line (already in %)
fig.add_trace(
    go.Scatter(
        x=df['Entry Time'], y=df['Drawdown'],
        mode='lines+markers',
        name='Drawdown (%)',
        line=dict(color='red'),
        text=df['Drawdown'].round(2).astype(str) + '%',
        hovertemplate='Date: %{x}<br>Drawdown: %{text}'
    ),
    row=2, col=1
)

fig.update_layout(
    height=700,
    width=1000,
    title_text="Portfolio Performance Overview",
    template='plotly_white'
)
fig.update_yaxes(title_text="Balance (₹)", row=1, col=1)
fig.update_yaxes(title_text="Drawdown (%)", row=2, col=1)
fig.update_xaxes(title_text="Time", row=2, col=1)

fig.show()

# ------------------------
# 2. Yearly Returns Histogram (%)
# ------------------------
df['Year'] = df['Entry Time'].dt.year
capital = 1000000  # base capital per trade
yearly_returns = df.groupby('Year')['PnL'].sum()
yearly_returns_pct = (yearly_returns / capital) * 100

fig_hist = go.Figure()
fig_hist.add_trace(go.Bar(
    x=yearly_returns_pct.index.astype(str),
    y=yearly_returns_pct.values,
    marker_color='green',
    name='Yearly Returns (%)',
    text=yearly_returns_pct.round(2).astype(str) + '%',
    textposition='outside',
    textfont=dict(size=12)
))

fig_hist.update_layout(
    title="Yearly Returns (%)",
    xaxis_title="Year",
    yaxis_title="Return (%)",
    template='plotly_white',
    width=800,
    height=400
)
fig_hist.show()

# ------------------------
# 3. Monthly Returns Heatmap (%)
# ------------------------
df['Month'] = df['Entry Time'].dt.month
monthly_returns = df.groupby(['Year', 'Month'])['PnL'].sum().unstack(fill_value=0)
monthly_returns_pct = (monthly_returns / capital) * 100

# Ensure all months are present
monthly_returns_pct = monthly_returns_pct.reindex(columns=range(1, 13), fill_value=0)
monthly_returns_pct.columns = [calendar.month_abbr[m] for m in monthly_returns_pct.columns]

# Heatmap with % numbers
heatmap = go.Figure(data=go.Heatmap(
    z=monthly_returns_pct.values,
    x=monthly_returns_pct.columns,
    y=monthly_returns_pct.index.astype(str),
    colorscale='RdBu',
    reversescale=True,
    colorbar_title='Return (%)',
    text=monthly_returns_pct.round(2).astype(str) + '%',
    texttemplate="%{text}",
    hovertemplate='Year: %{y}<br>Month: %{x}<br>Return: %{text}'
))

heatmap.update_layout(
    title="Monthly Returns Heatmap (%)",
    xaxis_title="Month",
    yaxis_title="Year",
    template='plotly_white',
    width=1000,
    height=500
)
heatmap.show()


In [ ]:
3import pandas as pd
import plotly.graph_objects as go
'''
# Step 1: Prepare data
df['Entry Time'] = pd.to_datetime(df['Entry Time'])
pnl_matrix = df.pivot_table(index='Entry Time', columns='Ticker', values='PnL')
pnl_matrix = pnl_matrix.dropna(thresh=10, axis=1).fillna(0)

# Step 2: Calculate cumulative PnL
cumulative_pnl = pnl_matrix.cumsum()

# Step 3: Normalize (start all at 0)
normalized = cumulative_pnl.subtract(cumulative_pnl.iloc[0])

# Step 4: Create colored line segments
fig = go.Figure()
tickers = normalized.columns

for ticker in tickers:
    times = normalized.index
    values = normalized[ticker].values

    # Draw colored segments between each pair of points
    for i in range(1, len(values)):
        color = 'green' if values[i] >= values[i-1] else 'red'
        fig.add_trace(go.Scatter(
            x=[times[i-1], times[i]],
            y=[values[i-1], values[i]],
            mode='lines',
            line=dict(color=color, width=2),
            name=ticker if i == 1 else None,  # Only label once
            showlegend=(i == 1)
        ))

# Final layout
fig.update_layout(
    title="Cumulative PnL of Tickers with Color-Coded Win/Loss Segments",
    xaxis_title="Time",
    yaxis_title="Normalized Cumulative PnL",
    height=500,
    width=1500
)

fig.show()
